In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from PIL import Image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class LensingSuperResDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, transform=None):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.transform = transform
        self.lr_images = sorted(os.listdir(lr_dir))
        self.hr_images = sorted(os.listdir(hr_dir))

    def __len__(self):
        return len(self.lr_images)

    def __getitem__(self, idx):
        lr_image = np.load(os.path.join(self.lr_dir, self.lr_images[idx]))
        hr_image = np.load(os.path.join(self.hr_dir, self.hr_images[idx]))

        lr_image = torch.tensor(lr_image, dtype=torch.float32).squeeze(0)
        hr_image = torch.tensor(hr_image, dtype=torch.float32).squeeze(0)

        # Add channel dimension (this could be the problem)
        lr_image = lr_image.unsqueeze(0)
        hr_image = hr_image.unsqueeze(0)

        if self.transform:
            lr_image = self.transform(lr_image)
            hr_image = self.transform(hr_image)

        return lr_image, hr_image

In [ ]:
lr_dir = "/content/drive/MyDrive/Dataset/LR"
hr_dir = "/content/drive/MyDrive/Dataset/HR"
batch_size = 16

In [ ]:
transform = transforms.Compose([transforms.Normalize(mean=[0.5], std=[0.5])])

In [ ]:
train_dataset = LensingSuperResDataset(lr_dir, hr_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Load Pre-trained Model
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=1, padding=0)
        self.conv3 = nn.Conv2d(32, 4, kernel_size=5, padding=2)
        self.upsample = nn.PixelShuffle(2)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.conv3(x)
        x = self.upsample(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SRCNN().to(device)

In [ ]:
# Load pre-trained weights (Task VI.A)
pretrained_weights = "/content/scrnn_weights3a"
model.load_state_dict(torch.load(pretrained_weights, map_location=device))

<All keys matched successfully>

In [ ]:
# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [1]:
# Training Loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for lr_imgs, hr_imgs in train_loader:
        lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)

        optimizer.zero_grad()
        sr_imgs = model(lr_imgs)  # Super-resolved output

        loss = criterion(sr_imgs, hr_imgs)  # MSE Loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.6f}")

NameError: name 'model' is not defined

In [ ]:
# Define the path where the model weights will be saved
model_path = "/content/weights6b.pth"

# Save Fine-Tuned Model
torch.save(model.state_dict(), model_path)
print("Fine-tuned model saved!")